# 实验03 正则表达式与网络结构化数据
学号：20244080318

实验环境：独立conda虚拟环境，禁止base环境
提交要求：
1. Markdown解释规则与结果；代码单元格完成计算，展示关键输出
2. 提交前重启内核，完整运行全部单元格
3. 仅提交：学号_姓名_exp03.ipynb


In [5]:
import hashlib
import os
import platform
import uuid
import sys

# 修改为自己的学号
STUDENT_ID = 20244080318
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]

RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]     {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")
print(f"[Python]   {sys.version.split()[0]}")


[Conda 环境] py_exp
[学号]     20244080318
[机器指纹] 86062bc25902fa85
[RUN_ID]   20244080318-86062bc25902fa85
[Python]   3.10.22


## 实验1：正则表达式的基础匹配

任务：导入re模块
1. 在`"cat on mat; later: cats and dogs"`分别匹配`\d+`、`\s+`、`\bcat\b`
2. 在`"cat dog bird CAT"`匹配`cat|dog`；对比`a|aa|aaa`与`aaa|aa|a`在`"aaaa"`上匹配差异
3. 在`"ct cat caat caaat"`测试量词：`ca*t`、`ca+t`、`ca?t`、`ca{2}t`、`ca{1,2}t`，输出匹配列表。

量词说明：
- `*`：前面字符0次或多次
- `+`：前面字符至少1次
- `?`：前面字符0次或1次
- `{n}`：固定n次
- `{m,n}`：m~n次


In [6]:
import re

text1 = "cat on mat; later: cats and dogs"
print(r"\d+", re.findall(r"\d+", text1))
print(r"\s+", re.findall(r"\s+", text1))
print(r"\bcat\b", re.findall(r"\bcat\b", text1))

text2 = "cat dog bird CAT"
print(r"cat|dog", re.findall(r"cat|dog", text2))
print(r"a|aa|aaa", re.findall(r"a|aa|aaa", "aaaa"))
print(r"aaa|aa|a", re.findall(r"aaa|aa|a", "aaaa"))

text3 = "ct cat caat caaat"
print(r"ca*t", re.findall(r"ca*t", text3))
print(r"ca+t", re.findall(r"ca+t", text3))
print(r"ca?t", re.findall(r"ca?t", text3))
print(r"ca{2}t", re.findall(r"ca{2}t", text3))
print(r"ca{1,2}t", re.findall(r"ca{1,2}t", text3))


\d+ []
\s+ [' ', ' ', ' ', ' ', ' ', ' ']
\bcat\b ['cat']
cat|dog ['cat', 'dog']
a|aa|aaa ['a', 'a', 'a', 'a']
aaa|aa|a ['aaa', 'a']
ca*t ['ct', 'cat', 'caat', 'caaat']
ca+t ['cat', 'caat', 'caaat']
ca?t ['ct', 'cat']
ca{2}t ['caat']
ca{1,2}t ['cat', 'caat']


## 实验2：分组、回溯引用与替换

任务：
1. 文本`"联系:alice@example.com;备选:bob@test.org"`，捕获组提取邮箱用户名、域名。
2. 文本`"cat cat, dog cat, bird bird"`，后向引用找出连续重复单词。
3. 使用`re.sub()`捕获组替换，将`2024-05-01\n2024/05/02\n2024.05.03`转换格式为`月‑日‑年`，展示转换前后。
4. 将`"联系我:123-456-7890 或 987.654.3210"`电话号码统一格式为`(123) 456-7890`。


In [7]:
import re

# 任务1 捕获组提取邮箱用户名、域名
mail_text = "联系:alice@example.com;备选:bob@test.org"
mail_pattern = r"([a-zA-Z0-9]+)@([a-zA-Z0-9.]+)"
mails = re.findall(mail_pattern, mail_text)
print("邮箱(用户名,域名):", mails)

# 任务2 后向引用匹配连续重复单词
dup_text = "cat cat, dog cat, bird bird"
dup_pattern = r"\b(\w+)\s+\1\b"
dup_res = re.findall(dup_pattern, dup_text)
print("连续重复单词：", dup_res)

# 任务3 日期格式转换
date_text = """2024-05-01
2024/05/02
2024.05.03"""
print("转换前：")
print(date_text)
date_pattern = r"(\d{4})[-/.](\d{2})[-/.](\d{2})"
date_out = re.sub(date_pattern, r"\2-\3-\1", date_text)
print("\n转换后：")
print(date_out)

# 任务4 电话号码统一格式
phone_text = "联系我:123-456-7890 或 987.654.3210"
phone_pattern = r"(\d{3})[-.](\d{3})[-.](\d{4})"
phone_out = re.sub(phone_pattern, r"(\1) \2-\3", phone_text)
print("\n电话号码转换结果：")
print(phone_out)


邮箱(用户名,域名): [('alice', 'example.com'), ('bob', 'test.org')]
连续重复单词： ['cat', 'bird']
转换前：
2024-05-01
2024/05/02
2024.05.03

转换后：
05-01-2024
05-02-2024
05-03-2024

电话号码转换结果：
联系我:(123) 456-7890 或 (987) 654-3210


## 实验3：使用BeautifulSoup解析HTML

原始html字符串：
```html
<html><head><title>实验课程</title></head><body> <h1>数据分析</h1> <p class="course">第六讲:正则表达式</p> <p class="course">第七讲:结构化数据</p> <a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a> </body></html>


In [8]:

from bs4 import BeautifulSoup

html_doc = """
<html><head><title>实验课程</title></head><body> <h1>数据分析</h1> <p class="course">第六讲:正则表达式</p> <p class="course">第七讲:结构化数据</p> <a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a> </body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")

# 任务1
title_tag = soup.find("title")
h1_tag = soup.find("h1")
print("页面标题文本：", title_tag.get_text())
print("h1标签名：", h1_tag.name)
print("h1文本：", h1_tag.get_text())

# 任务2
p_list = soup.find_all("p", class_="course")
print("\nclass=course的段落：")
for p in p_list:
    print(p.get_text())

a_list = soup.find_all("a")
print("\n全部链接：")
for a in a_list:
    print(f"链接文字:{a.get_text()}, href:{a['href']}")

# 任务3 get_text对比
print("\n整个文档get_text():")
print(repr(soup.get_text()))
print("\n单个p段落get_text():")
print(repr(p_list[0].get_text()))
print("\n单个a链接get_text():")
print(repr(a_list[0].get_text()))


页面标题文本： 实验课程
h1标签名： h1
h1文本： 数据分析

class=course的段落：
第六讲:正则表达式
第七讲:结构化数据

全部链接：
链接文字:讲义六, href:https://example.org/lecture06
链接文字:讲义七, href:https://example.org/lecture07

整个文档get_text():
'\n实验课程 数据分析 第六讲:正则表达式 第七讲:结构化数据 讲义六\n讲义七 \n'

单个p段落get_text():
'第六讲:正则表达式'

单个a链接get_text():
'讲义六'
